In [ ]:
import jax
import jax.numpy as jnp
import custom_jax as cj
import numpy as np
import fmdj

In [ ]:
x = jnp.array(([0.,0.,0.], [0.,0.,0.], [1.,0.,1.], [-1.,0.,-1.]))

print(cj.tree.build_ztree.jit(x)[0])
print("min,max level", -(127+23)*3, (128+1)*3)

x = jnp.array(([0.99,0.1,0.], [0.1,0.,0.1], [5.,0.1,1.1], [-1.,0.,-1.1]))
xdiff = x[1:] - x[:-1]
print(3*np.ceil(np.log2(np.abs(xdiff)))[:,0])

print(cj.tree.build_ztree.jit(x)[0])

# Compare with old tree 

In [ ]:
N = 1024*16

x3 = jax.random.uniform(jax.random.key(0), (N,3), jnp.float32, -0.5,0.5)
m3, _, isort1 = fmdj.octree.organize_particles.jit(x3, return_sorted=True)
x3sort, isort2 = cj.tree.pos_zorder_sort.jit(x3 + 0.5)

print(jnp.all(isort1 == isort2))

In [ ]:
level_binary = fmdj.octree.morton_diff_level(m3[1:], m3[:-1])

level, lbound, rbound, lchild, rchild = cj.tree.build_ztree.jit(x3sort)

print(jnp.all(-level[1:-1] == level_binary))

tree = fmdj.octree.get_compressed_binary_tree(m3)
print(jnp.all(tree.lbound == lbound))
print(jnp.all(tree.rbound == rbound))
print(jnp.all(tree.lchild == lchild))
print(jnp.all(tree.rchild == rchild))

# Profile

In [ ]:
N = 1024*1024*8

x3 = jax.random.uniform(jax.random.key(0), (N,3), jnp.float32, -0.5,0.5)

x3sort = cj.tree.pos_zorder_sort.jit(x3 + 0.5)[0].block_until_ready()
%timeit -n 20 cj.tree.pos_zorder_sort.jit(x3 + 0.5)[0].block_until_ready()
%timeit -n 20 cj.tree.build_ztree.jit(x3sort)[0].block_until_ready()

In [ ]:
from fmdj.utility import Timer

timer = Timer(print_compile=False, print_warmup=False)

for N in (int(1e4), int(3e4), int(1e5), int(3e5), int(1e6), int(3e6), int(1e7), int(3e7)):
    if N < 1e5:
        timer.loops = 1000
    elif N < 1e6:
        timer.loops = 100
    else:
        timer.loops = 20

    x3 = jax.random.uniform(jax.random.key(0), (N,3), jnp.float32, -0.5,0.5)
    x3s = x3 + 0.5
    x3sort = cj.tree.pos_zorder_sort.jit(x3 + 0.5)[0].block_until_ready()
    m3 = fmdj.octree.organize_particles.jit(x3, return_sorted=True)[0]
    timer.set_tag(N=N)

    
    timer.timeit_jit(fmdj.octree.organize_particles.jit, x3, return_sorted=False)
    timer.timeit_jit(fmdj.octree.get_compressed_binary_tree.jit, m3)

    timer.timeit_jit(cj.tree.pos_zorder_sort.jit, x3s)
    timer.timeit_jit(cj.tree.build_ztree.jit, x3sort)

In [ ]:
timer.plot_timings("N")